# 04 — Chains
### LangChain Foundations

`03_Runnables/` showed you the raw mechanics — `RunnableSequence`, `RunnableParallel`,
`RunnableBranch` — by building them and wiring them by hand. This notebook uses those exact same
pieces, but as **named, recognized patterns** you reach for directly when designing a real
pipeline: **simple**, **sequential**, **parallel**, and **conditional** chains.

The shift in mindset: last notebook asked *"what does `|` actually do?"* This notebook asks
*"given a real problem, which of these four shapes fits it?"*

## Setup

In [1]:
# %pip install -q langchain langchain-core langchain-groq python-dotenv pydantic
!pip install grandalf


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnablePassthrough
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Literal


model = ChatGroq(
    model="openai/gpt-oss-120b",
)
parser = StrOutputParser()

print("Environment ready.")


Environment ready.


---
## 1. Simple chain — and a debugging habit worth building now

`prompt | model | parser`. You've written this since `01_Models`. What's new here is
**`chain.get_graph().print_ascii()`** — every Runnable can draw its own shape. Get in the habit of
calling this on anything more complex than a single prompt; it's the fastest way to confirm a
pipeline is actually wired the way you think it is, before spending API calls debugging it blind.

In [3]:
prompt = PromptTemplate(
    template="Generate 3 interesting facts about {topic}.",
    input_variables=["topic"],
)

chain = prompt | model | parser

chain.get_graph().print_ascii()
print()
print(chain.invoke({"topic": "the ocean"}))


     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  

**Three fascinating ocean facts**

1. **The deep‑sea “rainforest” of bioluminescence** – Many deep‑water organisms (e.g., certain jellyfish, plankton, and fish) produce their own light. When a predator disturbs the water, a cascade of glowing 

---
## 2. Sequential chain — output of one step becomes the input of the next

Not "run A then B" in the sense of unrelated parallel work finishing at different times — this is
**B literally can't start until A's exact output is ready**, because B's input *is* A's output.
Classic use: generate something detailed, then compress it.

In [4]:
detail_prompt = PromptTemplate(
    template="Write a detailed, information-dense paragraph explaining {topic}.",
    input_variables=["topic"],
)
summary_prompt = PromptTemplate(
    template="Summarize the following into exactly 5 bullet points:\n\n{text}",
    input_variables=["text"],
)

sequential_chain = detail_prompt | model | parser | summary_prompt | model | parser

sequential_chain.get_graph().print_ascii()
print()
print(sequential_chain.invoke({"topic": "how vaccines work"}))


     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

Notice the chain is 6 steps long (`prompt → model → parser → prompt → model → parser`) but it's
still just `RunnableSequence` all the way down — nothing new mechanically versus `03_Runnables`,
just a longer pipe. The output of the first `parser` becomes the `{text}` fed into `summary_prompt`
automatically, because `RunnableSequence.invoke()` always feeds one step's output into the next.

---
## 3. Parallel chain — independent branches, merged into one result

From `03_Runnables` you know *why* this is faster than sequential (nothing here depends on
anything else here). The new part is the **real-world shape**: generate two genuinely different
artifacts from the same source material, then combine them into one deliverable with a third,
merging prompt.

In [5]:
SAMPLE_TEXT = """
Photosynthesis is the process by which plants, algae, and some bacteria convert light energy,
usually from the sun, into chemical energy stored in glucose. It occurs mainly in the chloroplasts
of plant cells, using a green pigment called chlorophyll to absorb light. The process has two main
stages: the light-dependent reactions, which occur in the thylakoid membrane and produce ATP and
NADPH while releasing oxygen as a byproduct, and the light-independent reactions (the Calvin cycle),
which occur in the stroma and use ATP and NADPH to convert carbon dioxide into glucose. Photosynthesis
is essential for life on Earth, as it produces the oxygen most organisms breathe and forms the base
of nearly every food chain.
"""

notes_prompt = PromptTemplate(
    template="""You are an expert educator. Convert the text below into well-structured study notes.
Use Markdown headings and bullet points. Cover every important concept. No paragraph summary.

Text:
{text}""",
    input_variables=["text"],
)

quiz_prompt = PromptTemplate(
    template="""You are an expert quiz creator. Based on the text below, write 3 multiple-choice
questions, each with 4 options (A-D) and the correct answer marked at the end.

Text:
{text}""",
    input_variables=["text"],
)

parallel_chain = RunnableParallel(
    notes=notes_prompt | model | parser,
    quiz=quiz_prompt | model | parser,
)

parallel_chain.get_graph().print_ascii()


          +---------------------------+            
          | Parallel<notes,quiz>Input |            
          +---------------------------+            
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

In [6]:
result = parallel_chain.invoke({"text": SAMPLE_TEXT})
print("--- NOTES ---")
print(result["notes"][:300], "...\n")
print("--- QUIZ ---")
print(result["quiz"][:300], "...")


--- NOTES ---
## Photosynthesis Overview
- **Definition:** Conversion of light energy into chemical energy (glucose) by plants, algae, and certain bacteria.  
- **Location:** Primarily in chloroplasts of plant cells.  
- **Primary pigment:** Chlorophyll (green) absorbs light.

## Chloroplast Structure Relevant to ...

--- QUIZ ---
**Quiz – Photosynthesis**

---

**1. Where do the light‑dependent reactions of photosynthesis take place?**  
A. Stroma of the chloroplast  
B. Cytoplasm of the plant cell  
C. Thylakoid membrane  
D. Mitochondrial inner membrane  

**Answer: C**

---

**2. Which molecule is the primary pigment that ...


---
## 4. Merging parallel results — a third prompt that combines both branches

`parallel_chain` above returns a dict. Pipe that dict straight into one more prompt that expects
both keys, and you get a single combined document out of two independently-generated pieces.

In [7]:
merge_prompt = PromptTemplate(
    template="""Combine the following into a single study guide.
Include ALL notes exactly as given, then a "## Quiz" heading, then ALL quiz questions exactly as given.
Do not summarize or remove anything.

# Notes
{notes}

# Quiz
{quiz}""",
    input_variables=["notes", "quiz"],
)

study_guide_chain = parallel_chain | merge_prompt | model | parser

study_guide_chain.get_graph().print_ascii()


          +---------------------------+            
          | Parallel<notes,quiz>Input |            
          +---------------------------+            
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

---
## 5. Conditional chain — routing on a *reliable* signal, not raw text

`RunnableBranch`'s condition functions need something dependable to check — and raw model text
output ("the language is definitely Hindi!") is exactly the kind of thing that's inconsistent
enough to break a routing decision. The fix: force the classification step to return a **structured**
value instead of free text, using `PydanticOutputParser`.

> This is a light preview — `05_Structured_Output/` covers structured output properly, including
> the more modern `.with_structured_output()` method. Here, we're using just enough of it to make
> `RunnableBranch`'s routing decision reliable.

In [8]:
class LanguageDetection(BaseModel):
    language: Literal["English", "Hindi", "Other"] = Field(
        description="The detected language of the input text."
    )

structured_parser = PydanticOutputParser(pydantic_object=LanguageDetection)

detect_prompt = PromptTemplate(
    template="""Identify the language of the text below. Respond in the exact format requested.

Text:
{text}

{format_instructions}""",
    input_variables=["text"],
    partial_variables={"format_instructions": structured_parser.get_format_instructions()},
)

detect_chain = detect_prompt | model | structured_parser

result = detect_chain.invoke({"text": "Yeh ek Hindi vaakya hai."})
print(result)               # a real LanguageDetection object, not a string to parse yourself
print(result.language)      # -> reliable, always one of exactly three values


language='Hindi'
Hindi


In [9]:
to_hindi_prompt = PromptTemplate(
    template="Translate the following English text into natural, fluent Hindi. Return only the translation.\n\n{text}",
    input_variables=["text"],
)
to_english_prompt = PromptTemplate(
    template="Translate the following Hindi text into natural, fluent English. Return only the translation.\n\n{text}",
    input_variables=["text"],
)
unsupported_prompt = PromptTemplate(
    template="Politely tell the user, in one sentence, that this translator currently only supports English and Hindi.",
    input_variables=[],
)

translate_branch = RunnableBranch(
    (lambda x: x["language"] == "English", to_hindi_prompt | model | parser),
    (lambda x: x["language"] == "Hindi",   to_english_prompt | model | parser),
    unsupported_prompt | model | parser,   # default
)

# detect_chain returns a LanguageDetection object; translate_branch needs a dict with "text" + "language"
translation_pipeline = RunnablePassthrough.assign(
    language=lambda x: detect_chain.invoke({"text": x["text"]}).language
) | translate_branch

print(translation_pipeline.invoke({"text": "The sun rises in the east."}))
print(translation_pipeline.invoke({"text": "Yeh ek Hindi vaakya hai."}))
print(translation_pipeline.invoke({"text": "Ceci est une phrase française."}))


सूरज पूर्व में उगता है।
This is a Hindi sentence.
I’m sorry, but this translator currently supports only English and Hindi.


---
## Real-world capstone: study guide generator with translation

Every pattern in this notebook, combined into one tool: given a block of source text and a target
language, produce a merged notes+quiz study guide — then, only if the user asked for Hindi,
translate the finished guide before returning it. Sequential, parallel, and conditional, in one
pipeline, driven by a single `.invoke()` call.

In [10]:
full_pipeline = (
    RunnablePassthrough.assign(guide=study_guide_chain)   # keeps "text" AND "target_language" alive
    | RunnableBranch(
        (lambda x: x["target_language"] == "Hindi",
         (lambda x: {"text": x["guide"]}) | to_hindi_prompt | model | parser),
        lambda x: x["guide"],   # default: no translation needed, return the guide as-is
    )
)

english_guide = full_pipeline.invoke({"text": SAMPLE_TEXT, "target_language": "English"})
print(english_guide[:400], "...\n")

hindi_guide = full_pipeline.invoke({"text": SAMPLE_TEXT, "target_language": "Hindi"})
print(hindi_guide[:400], "...")


# Photosynthesis Study Notes  

## 1. General Overview  
- **Definition:** Conversion of light energy (usually solar) into chemical energy stored as glucose.  
- **Organisms:** Plants, algae, and certain bacteria.  
- **Location:** Primarily in chloroplasts of plant cells.  

## 2. Key Pigment  
- **Chlorophyll** – green pigment that absorbs light energy and initiates the photosynthetic process.   ...

# प्रकाशसंश्लेषण अध्ययन नोट्स  

## 1. सामान्य परिचय  
- **परिभाषा:** प्रकाश ऊर्जा को रासायनिक ऊर्जा (ग्लूकोज़) में परिवर्तित करना।  
- **संलग्न जीव:**  
  - पौधे  
  - शैवाल  
  - कुछ बैक्टीरिया (प्रकाशसंश्लेषी बैक्टीरिया)  
- **ऊर्जा स्रोत:** मुख्यतः सूर्य का प्रकाश।  

## 2. प्रकाशसंश्लेषण का स्थान  
- **अवयव:** क्लोरोप्लास्ट (पौधा कोशिकाओं में)।  
- **मुख्य रंगद्रव्य:** क्लोरोफिल (हरा) – प्रका ...


In [11]:
# Save the result the same way you would in a real tool
with open("study_guide.md", "w", encoding="utf-8") as f:
    f.write(english_guide)

print("Saved to study_guide.md")


Saved to study_guide.md


### Try it yourself
1. Swap `SAMPLE_TEXT` for a paragraph on a topic you actually care about and regenerate the guide.
2. Add a third target language option and a matching translation branch.
3. Call `full_pipeline.get_graph().print_ascii()` before running it — predict what you'll see, then
   check whether you were right.
4. `parallel_chain` currently ignores errors from either branch — deliberately break `quiz_prompt`
   (typo a variable name) and observe what happens to the whole pipeline when one parallel branch
   fails. `16_Error_Handling/`, later in this repo, covers how to make chains resilient to exactly this.

---
**Next:** `04.5_Output_Parsers/` (or `05_Structured_Output/`, depending on how you've ordered your
repo) — the `PydanticOutputParser` you just used to make a branch condition reliable gets the full
treatment: every parser type, when to use which, and the newer `.with_structured_output()` method.
